In [4]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

C:\Users\jacob\AppData\Roaming\Python\Python313\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [5]:
df=pd.read_csv("insurance_data.csv")
df
df.head()

,age,affordability,have_insurance
0,22,1,0
1,25,0,0
2,47,1,1
3,52,0,0
4,46,1,1


In [30]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(df[['age','affordability']],df.have_insurance,test_size=0.2,random_state=25)

In [31]:
x_train

,age,affordability
1,25,0
9,61,1
12,27,0
0,22,1
5,56,1
2,47,1
7,60,0
6,55,0
10,18,1
4,46,1


In [10]:
x_test

,age,affordability
11,28,1
8,62,1
3,52,0


In [12]:
y_train

1     0
9     1
12    1
0     0
5     1
2     1
7     1
6     0
10    0
4     1
Name: have_insurance, dtype: int64

In [7]:
y_test

11    0
8     1
3     0
Name: have_insurance, dtype: int64

In [12]:
x_train_scaled=x_train.copy()
x_train_scaled['age']=x_train_scaled['age']/100
x_test_scaled=x_test.copy()
x_test_scaled['age']=x_test_scaled['age']/100


In [17]:
x_train_scaled

,age,affordability
1,0.25,0
9,0.61,1
12,0.27,0
0,0.22,1
5,0.56,1
2,0.47,1
7,0.60,0
6,0.55,0
10,0.18,1
4,0.46,1


In [13]:
model=keras.Sequential([
    keras.layers.Dense(1,input_shape=(2,),activation='sigmoid',kernel_initializer='ones',bias_initializer='zeros')
])
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
model.fit(x_train_scaled,y_train,epochs=100)


Epoch 1/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 717ms/step - accuracy: 0.6000 - loss: 0.6552
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.6000 - loss: 0.6549
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.6000 - loss: 0.6547
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.6000 - loss: 0.6545
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.6000 - loss: 0.6543
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.6000 - loss: 0.6541
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.6000 - loss: 0.6539
Epoch 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.6000 - loss: 0.6537
Epoch 9/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.6000 - loss: 0.6535
Epoch 10/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - accuracy: 0.6000 - loss: 0.6533
Epoch 11/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.6000 - loss: 0.6530
Epoch 12/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.6000 - l

In [22]:
model.evaluate(x_test_scaled,y_test)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 819ms/step - accuracy: 0.3333 - loss: 0.6311


[0.6311001181602478, 0.3333333432674408]

In [14]:

model.predict(x_test_scaled)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step


array([[0.7461817],
       [0.8028095],
       [0.5994468]], dtype=float32)

In [8]:
y_test

11    0
8     1
3     0
Name: have_insurance, dtype: int64

In [15]:
#weight and bias
#tensorflow calculation
coef,intercept=model.get_weights()
coef,intercept

(array([[0.9576369],
        [0.9050229]], dtype=float32),
 array([-0.09481066], dtype=float32))

In [16]:
#manual calculation
def sigmoid(x):
    import math
    return 1/(1+math.exp(-x))
sigmoid(18)    
    

0.9999999847700205

In [58]:
def prediction_function(age,affordability):
    weighted_sum=coef[0]*age+coef[1]*affordability+intercept
    
    return sigmoid(weighted_sum)
    

In [34]:
prediction_function(0.47,1)

C:\Users\jacob\AppData\Local\Temp\ipykernel_13128\2495765867.py:3: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return 1/(1+math.exp(-x))


0.7790780204440064

In [52]:
def sigmoid_numpy(X):
   return 1/(1+np.exp(-X))

sigmoid_numpy(np.array([12,0,1]))

array([0.99999386, 0.5       , 0.73105858])

In [70]:
def log_loss(y_true, y_predicted):
    epsilon = 1e-15
    y_predicted_new = [max(i,epsilon) for i in y_predicted]
    y_predicted_new = [min(i,1-epsilon) for i in y_predicted_new]
    y_predicted_new = np.array(y_predicted_new)
    return -np.mean(y_true*np.log(y_predicted_new)+(1-y_true)*np.log(1-y_predicted_new))

In [71]:
def gradient_descent(age, affordability, y_true, epochs, loss_thresold):
    w1 = w2 = 1
    bias = 0
    rate = 0.5
    n = len(age)
    for i in range(epochs):
        weighted_sum = w1 * age + w2 * affordability + bias
        y_predicted = sigmoid_numpy(weighted_sum)
        loss = log_loss(y_true, y_predicted)

        w1d = (1/n)*np.dot(np.transpose(age),(y_predicted-y_true)) 
        w2d = (1/n)*np.dot(np.transpose(affordability),(y_predicted-y_true)) 

        bias_d = np.mean(y_predicted-y_true)
        w1 = w1 - rate * w1d
        w2 = w2 - rate * w2d
        bias = bias - rate * bias_d

        print (f'Epoch:{i}, w1:{w1}, w2:{w2}, bias:{bias}, loss:{loss}')

        if loss<=loss_thresold:
            break

    return w1, w2, bias
            

In [72]:
gradient_descent(x_train_scaled['age'],x_train_scaled['affordability'],y_train,1000,0.4631)

Epoch:0, w1:0.9953023510435051, w2:0.9589316923292891, bias:-0.061521372825103324, loss:0.6551533000097722
Epoch:1, w1:0.993910363508503, w2:0.9229449877737186, bias:-0.11492051744354004, loss:0.6448842970323124
Epoch:2, w1:0.9954617896524035, w2:0.891582187553737, bias:-0.16108305206320225, loss:0.6371406708513416
Epoch:3, w1:0.9995943354400736, w2:0.8643542574577828, bias:-0.2008910331891214, loss:0.631323431585787
Epoch:4, w1:1.0059608511098919, w2:0.8407710950196449, bias:-0.23518694061777196, loss:0.6269452988730868
Epoch:5, w1:1.0142393890659045, w2:0.8203633699084859, bias:-0.2647501257590066, loss:0.623623723434239
Epoch:6, w1:1.0241388214587746, w2:0.8026964734931715, bias:-0.2902839303638856, loss:0.6210669192714671
Epoch:7, w1:1.0354010237980271, w2:0.7873780043929964, bias:-0.3124109840420571, loss:0.6190574546564643
Epoch:8, w1:1.0478006423445003, w2:0.7740604570653357, bias:-0.3316742051530844, loss:0.6174363139122309
Epoch:9, w1:1.0611433166350694, w2:0.7624406458419507,

(np.float64(7.762790740073865),
 np.float64(0.8757995704927221),
 np.float64(-3.2050626291525384))